# EKPO: purchasing document items (EDA)

One row per document item. Holds the values (Net Order Value), quantities, material, material group
and receiving plant.

Raw export: 3,974 rows x 418 columns; `sap_data.load` keeps 16.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sap_data import load, load_raw, PURCH_ORG, COMPANY_CODE, PERIOD_START, PERIOD_END

pd.set_option("display.max_columns", 50)
sns.set_theme(style="whitegrid")

In [ ]:
ekpo = load("ekpo")
ekko = load("ekko")
print(ekpo.shape)
display(ekpo.head())
ekpo.dtypes

## 1. Grain and link to EKKO

In [ ]:
assert not ekpo.duplicated(["Purchasing Document", "Item"]).any()
print("Items without header:", (~ekpo["Purchasing Document"].isin(ekko["Purchasing Document"])).sum())
po_hdr = ekko.loc[ekko["Purch. Doc. Category"] == "F", "Purchasing Document"]
print("PO headers without items:", (~po_hdr.isin(ekpo["Purchasing Document"])).sum())

# Category is stored on both levels; check they agree
chk = ekpo.merge(ekko[["Purchasing Document", "Purch. Doc. Category"]],
                 on="Purchasing Document", suffixes=("", "_hdr"), validate="many_to_one")
print("Category mismatches item vs header:",
      (chk["Purch. Doc. Category"] != chk["Purch. Doc. Category_hdr"]).sum())
ekpo.groupby("Purchasing Document").size().describe()

> **Note.** Grain confirmed: (`Purchasing Document`, `Item`) is unique. Every item has a header, and the
> item-level category always matches the header. 38 PO headers have no items (empty or abandoned POs).
> Most documents have a single item, so PO count and PO item count are close but **not** equal.

## 2. Missing values

In [ ]:
ekpo.isna().agg(["sum", "mean"]).T.sort_values("sum", ascending=False)

> **Note.** Only `Deletion Indicator` is mostly empty, which is expected because blank means "not
> deleted". A few items lack plant, material group or material.

## 3. Value by document category

In [ ]:
ekpo.groupby("Purch. Doc. Category").agg(
    items=("Item", "size"),
    net_value=("Net Order Value", "sum"),
    zero_value_items=("Net Order Value", lambda s: (s == 0).sum()),
    deleted=("Deletion Indicator", lambda s: s.eq("L").sum()))

> **Note.** RFQ (A) and contract (K) items always have a Net Order Value of 0, which confirms that only
> category F is ordered value. 64 items are flagged deleted (`L`), 5 of them on POs.
> **Decision:** exclude `Deletion Indicator == "L"`.

## 4. Net Order Value on PO items

In [ ]:
pf = ekpo[(ekpo["Purch. Doc. Category"] == "F") & ekpo["Deletion Indicator"].isna()].copy()
display(pf[["Order Quantity", "Net Order Price", "Price unit", "Net Order Value"]].describe().T)
print("Zero-value items:", (pf["Net Order Value"] == 0).sum())
print("Price unit == 0:", (pf["Price unit"] == 0).sum())

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 3.5))
sns.histplot(pf["Net Order Value"].clip(lower=1), log_scale=True, bins=40, ax=ax[0])
ax[0].set(title="Net PO value per item (all dates, log scale)", xlabel="Net PO value, USD")
sns.boxplot(x=pf["Net Order Value"].clip(lower=1), log_scale=True, ax=ax[1])
ax[1].set(title="Same, as boxplot", xlabel="Net PO value, USD");

In [ ]:
pf.nlargest(10, "Net Order Value")[["Purchasing Document", "Item", "Short Text", "Material Group",
                                    "Order Quantity", "Net Order Price", "Net Order Value"]]

> **Note.** The distribution is extremely right-skewed. The median is a few thousand USD, and a single
> item, PO 4500001630 (200,100 chain locks x $32 = **$6.4M**, Oct 2026), is about 47% of all PO value in
> the extract. We report medians next to means and use log scales. That outlier falls outside the
> reporting window, but it should be mentioned as a limitation.

### Value consistency check
`Net Order Value` should equal `Order Quantity × Net Order Price / Price unit`.

In [ ]:
pu0 = pf["Price unit"] == 0
print("Price unit = 0 (value can't be recomputed):", pu0.sum(), "items, net value", pf.loc[pu0, "Net Order Value"].sum())
calc = pf["Order Quantity"] * pf["Net Order Price"] / pf["Price unit"]
bad = pf[~pu0 & ((calc - pf["Net Order Value"]).abs() > 0.01)]
print(len(bad), "other items do not reconcile, net value", bad["Net Order Value"].sum())
bad[["Purchasing Document", "Item", "Order Quantity", "Net Order Price", "Price unit", "Net Order Value"]].head(10)

> **Note.** 54 PO items have `Price unit = 0`. They carry a price, but SAP stored a value of 0, which
> looks like an exercise entry error. Another ~12 items don't reconcile, mostly with quantity 0 and
> a value equal to one unit price. **Decision:** we trust `Net Order Value` as SAP calculated it and
> don't recompute it. All of these items together are worth only a few hundred USD.

## 5. Categorical attributes (PO items)

In [ ]:
for c in ["Material Group", "Material Type", "Plant", "Order Unit"]:
    display(pf.groupby(c, dropna=False)["Net Order Value"].agg(items="size", net_value="sum")
              .assign(share=lambda d: d.net_value / d.net_value.sum())
              .sort_values("net_value", ascending=False).round(3))

> **Note.** Material groups are coarse. `UTIL` (trading goods / accessories) and `RAW` (raw materials,
> mostly bolts) dominate. `BIKES`, `WHEEL` and `SFTY` have only a handful of items. Plants are
> **MI00** (Miami) and **DL00** (Dallas), plus 4 items for SD00 (San Diego). Order unit is almost
> always EA, with a few KG (flour items, see below).

## 6. What is actually being bought: material and short text

In [ ]:
pf["text_norm"] = pf["Short Text"].str.lower().str.strip()
(pf.groupby(["Material Group", "text_norm"], dropna=False)["Net Order Value"]
   .agg(items="size", net_value="sum").sort_values("net_value", ascending=False).head(25))

In [ ]:
pf["Material"].str[:2].value_counts().head(15)

In [ ]:
pf.loc[pf["text_norm"].str.contains("chain lo", na=False), "text_norm"].value_counts().head(15)

> **Note.** The real material mix is narrow: **chain locks** (spelled many ways: "chain lock",
> "chain lokk", "chain lock 108", ...), **socket head bolts**, **water bottles**, t-shirts, plus a
> student exercise item "Tönkölybúza liszt" (spelt flour) booked under UTIL/RAW at high prices. The GBI
> material prefix (`CH` chain lock, `BO` bolt, `TK` flour, `SH` shirt, `CA` cage, …) is a more reliable
> category key than the free-text Short Text. This matters for Q4: the "Frames/Wheels/Off-road"
> categories in the question barely exist in this data.

## Summary for the report (EKPO)
- Grain: 1 row per document item; join to EKKO on `Purchasing Document` (many-to-one, all items match).
- Value measure: `Net Order Value` on category F items, excluding deletion indicator `L`.
- Heavy right skew and one $6.4M outlier (Oct 2026). Use median, log scales, and state the window.
- 76 zero-value PO items (54 with price unit 0) and ~12 non-reconciling items stay in at ~$0.
- Material group is coarse (UTIL/RAW dominate). Material prefix or normalised text is needed for a
  meaningful category view (Q4).

## Appendix: columns left out

In [ ]:
raw = load_raw("ekpo")
informative = [c for c in raw.columns if raw[c].nunique(dropna=True) > 1]
print(f"{raw.shape[1]} columns, {len(informative)} with more than one distinct value:")
print(informative)